# M3-B1 — Fusion de deux sources + colonne de provenance

> ~30-40 min. Le geste **exact** que le cas d'usage certif attend côté
> données : réunir **deux exports comparables** (même métier, deux
> origines) dans un seul tableau, **sans perdre d'où vient chaque ligne**.

Ici : deux sites d'Acerox t'envoient chacun un export de capteurs. Même
idée, mais ce **ne sont pas** exactement les mêmes fichiers. Ton travail :
les empiler proprement et repérer ce qui cloche.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path('..') / 'data'
a = pd.read_csv(DATA_DIR / 'capteurs_site_A.csv')
b = pd.read_csv(DATA_DIR / 'capteurs_site_B.csv')
print('A', a.shape, '| colonnes', list(a.columns))
print('B', b.shape, '| colonnes', list(b.columns))

A (25, 5) | colonnes ['ts', 'machine_id', 'temp_c', 'vibration_mm_s', 'debit_l_min']
B (22, 5) | colonnes ['ts', 'machine_id', 'temp_c', 'vibration_mm_s', 'firmware']


## 1. Le réflexe naïf (à ne PAS garder)

On empile les deux sans réfléchir. Regarde bien le résultat.

In [2]:
naif = pd.concat([a, b], ignore_index=True)
naif.info()
display(naif.head())
display(naif.tail())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ts              47 non-null     object 
 1   machine_id      47 non-null     object 
 2   temp_c          47 non-null     float64
 3   vibration_mm_s  47 non-null     float64
 4   debit_l_min     25 non-null     float64
 5   firmware        22 non-null     object 
dtypes: float64(3), object(3)
memory usage: 2.3+ KB


,ts,machine_id,temp_c,vibration_mm_s,debit_l_min,firmware
0,2026-06-01T06:00:00,M01,72.5,1.77,124.7,NaN
1,2026-06-01T07:00:00,M04,69.9,2.36,125.7,NaN
2,2026-06-01T08:00:00,M04,64.6,2.40,126.3,NaN
3,2026-06-01T09:00:00,M03,69.5,2.32,117.2,NaN
4,2026-06-01T10:00:00,M03,64.2,1.83,116.3,NaN


,ts,machine_id,temp_c,vibration_mm_s,debit_l_min,firmware
42,2026-06-01T23:00:00,M05,156.6,3.30,NaN,v1.2
43,2026-06-02T00:00:00,M04,155.2,3.76,NaN,v1.3
44,2026-06-02T01:00:00,M05,166.2,2.51,NaN,v1.2
45,2026-06-02T02:00:00,M08,153.3,1.81,NaN,v1.2
46,2026-06-02T03:00:00,M05,147.8,1.23,NaN,v1.3


### Ce qui vient de casser

* Après le `pd.concat`, il est impossible de savoir si une ligne vient du site A ou du site B, car aucune colonne de provenance n’a été conservée.
* La colonne `debit_l_min` contient 22 valeurs manquantes, car elle existe uniquement dans l’export du site A. Le site B fournit à la place une colonne `firmware`.
* La colonne `firmware` contient 25 valeurs manquantes pour la même raison : elle n’existe pas dans l’export du site A.
* La colonne `temp_c` mélange probablement deux unités différentes. Les valeurs du site A sont proches de 65–75, tandis que celles du site B sont proches de 145–166, ce qui suggère que le site B utilise des degrés Fahrenheit malgré le nom de colonne `temp_c`.
* Certains identifiants de machines peuvent également exister dans les deux sites. Sans provenance, un identifiant comme `M04` peut devenir ambigu.

Cette fusion naïve conserve les lignes, mais elle perd le contexte nécessaire pour interpréter correctement les données.


## 2. Le bon geste : marquer la provenance AVANT de concaténer

Une colonne `source` = une **fiche d'identité** de chaque ligne. C'est la
base de la traçabilité (RGPD, réconciliation, filtrage par scénario ensuite).

In [ ]:
a2 = a.assign(source='site_A')
b2 = b.assign(source='site_B')
fusion = pd.concat([a2, b2], ignore_index=True)
fusion['source'].value_counts()

## 3. Maintenant, la provenance te fait VOIR les pièges

Fusionner n'est pas empiler. Trois choses à débusquer avec la colonne `source`.

In [ ]:
# Piège A — mêmes machine_id des deux côtés : le « M04 » de A est-il
# le même équipement que le « M04 » de B ? (indice : non)
communs = set(a['machine_id']) & set(b['machine_id'])
print('machine_id présents dans les DEUX sites :', sorted(communs))
# TODO — que faire ? préfixer par la source ? (ex. 'site_A::M04')

In [ ]:
# Piège B — même colonne temp_c, mais l'échelle diffère selon la source.
fusion.groupby('source')['temp_c'].describe()[['mean', 'min', 'max']]
# TODO — une des deux sources est en °F (unité oubliée). Laquelle ?
# Sans la colonne `source`, cette anomalie était invisible.

In [ ]:
# Piège C — debit_l_min n'existe que pour un site -> NaN pour l'autre.
fusion.groupby('source')['debit_l_min'].apply(lambda s: s.isna().mean())
# TODO — colonne réellement absente, ou juste pas transmise ? -> question client.

## Ce que tu dois en retenir

- **Fusionner ≠ empiler.** Deux fichiers « du même genre » cachent des
  écarts de schéma, d'unité et de clé.
- La colonne **`source` (provenance)** n'est pas cosmétique : c'est elle
  qui rend les écarts **visibles** et traçables — et qui te laissera plus
  tard filtrer par origine (scénarios, RGPD, réconciliation).
- Reporte une phrase dans `identification_sources.md` : *« Les sources X
  et Y ont été réunies avec une colonne `source` ; écarts constatés : … »*